# 30: Gold manifest

Builds a training release in `gold.manifest_rows` from the file in `config/gold/manifests/` named by `MANIFEST_NAME`. To build a different release, add a new YAML file there and change `MANIFEST_NAME`.

Run after every included dataset's `20_silver_validate`. The release records the git commit of this code, and the notebook refuses to run without one.

In [ ]:
%run ./_setup_env


In [ ]:
from data_platform.provenance import resolve_git_commit
from data_platform.sampling import select_sample_and_splits
from data_platform.dataset_layout import load_yaml_config
from data_platform.spark_io import (
    assert_no_cross_dataset_duplicate_checksums,
    build_gold_candidate_groups,
    print_gold_outputs,
    write_gold_release,
)

# Shared tables, the same for every dataset.
SILVER_TABLES = {
    "image_inventory": "silver.image_inventory",
    "leakage_groups": "silver.leakage_groups",
    "rejected_records": "silver.rejected_records",
}
GOLD_TABLES = {"manifest_rows": "gold.manifest_rows"}

# Which release to build: config/gold/manifests/<release_name>.yaml. Set by the build_release
# job's parameters; a manual run uses these defaults.
dbutils.widgets.text("release_name", "sample-v1")
dbutils.widgets.text("overwrite_existing_release", "false")
dbutils.widgets.text("git_commit", "")
MANIFEST_NAME = dbutils.widgets.get("release_name")
MANIFEST_CONFIG = load_yaml_config(CONFIG_ROOT / "gold" / "manifests" / f"{MANIFEST_NAME}.yaml")

DATASET_KEYS = MANIFEST_CONFIG["dataset_keys"]
LABEL_COLUMN = MANIFEST_CONFIG["label_column"]
SAMPLE_SIZE_PER_DATASET = MANIFEST_CONFIG["sample_size_per_dataset"]
# sample_seed picks the images, split_seed divides them into splits (see the manifest config).
SAMPLE_SEED = MANIFEST_CONFIG["sample_seed"]
SPLIT_RATIOS = tuple(MANIFEST_CONFIG["split_ratios"])
SPLIT_SEED = MANIFEST_CONFIG["split_seed"]
DATASET_VERSION = MANIFEST_CONFIG["dataset_version"]
PREPROCESSING_VERSION = MANIFEST_CONFIG["preprocessing_version"]

# Published releases are immutable: rebuilding an existing dataset_version fails unless this is
# True. Prefer a new dataset_version in the manifest config over overwriting one models may use.
OVERWRITE_EXISTING_RELEASE = dbutils.widgets.get("overwrite_existing_release").strip().lower() == "true"

# The commit of the code writing this release, resolved up front so a missing commit fails
# before any work is done (see data_platform.provenance).
GIT_COMMIT = resolve_git_commit(REPO_ROOT, explicit_commit=dbutils.widgets.get("git_commit"))
print(f"git_commit: {GIT_COMMIT}")


## Summarize each dataset into leakage groups

A dataset with no accepted rows for the label column yet is skipped with a warning.

In [ ]:
candidate_groups_by_dataset = {}
for dataset_key in DATASET_KEYS:
    groups = build_gold_candidate_groups(spark, SILVER_TABLES, dataset_key, LABEL_COLUMN)
    if not groups:
        print(f"{dataset_key}: no accepted rows with {LABEL_COLUMN} yet -- skipping (Silver not run for this dataset yet)")
        continue
    candidate_groups_by_dataset[dataset_key] = groups
    print(f"{dataset_key}: {len(groups)} candidate leakage groups")


## Check for duplicate images across datasets

Fails if the same image (by checksum) is in more than one of the datasets being combined ([ADR 003](../docs/decisions/003-cross-dataset-leakage-not-checked.md)).

In [ ]:
assert_no_cross_dataset_duplicate_checksums(spark, SILVER_TABLES, list(candidate_groups_by_dataset))
print(f"No cross-dataset duplicate images found among {list(candidate_groups_by_dataset)}")


## Sample whole groups and assign splits

Seeded and deterministic: the same config always gives the same release.

In [ ]:
split_by_group_by_dataset = {
    dataset_key: select_sample_and_splits(
        groups,
        sample_size=SAMPLE_SIZE_PER_DATASET,
        sample_seed=SAMPLE_SEED,
        split_seed=SPLIT_SEED,
        split_ratios=SPLIT_RATIOS,
    )
    for dataset_key, groups in candidate_groups_by_dataset.items()
}
for dataset_key, split_by_group in split_by_group_by_dataset.items():
    image_count = sum(
        g["image_count"] for g in candidate_groups_by_dataset[dataset_key] if g["group_id"] in split_by_group
    )
    print(f"{dataset_key}: {len(split_by_group)} groups selected, {image_count} images")


## Write the release

Writes every dataset's rows in one atomic replace of this `dataset_version`. If the release already exists, this stops unless `OVERWRITE_EXISTING_RELEASE = True`; published releases are meant to stay fixed.

In [ ]:
write_gold_release(
    spark,
    SILVER_TABLES,
    GOLD_TABLES,
    split_by_group_by_dataset,
    label_column=LABEL_COLUMN,
    dataset_version=DATASET_VERSION,
    sample_seed=SAMPLE_SEED,
    split_seed=SPLIT_SEED,
    preprocessing_version=PREPROCESSING_VERSION,
    git_commit=GIT_COMMIT,
    overwrite_existing=OVERWRITE_EXISTING_RELEASE,
)


## Review

Fails if any leakage group ended up in more than one split.

In [ ]:
print_gold_outputs(
    spark, display, GOLD_TABLES, DATASET_VERSION, list(split_by_group_by_dataset), LABEL_COLUMN,
)


## Registry entry for this release

In [ ]:
display(spark.sql(f"SELECT * FROM gold.manifest_registry WHERE dataset_version = '{DATASET_VERSION}'"))
